# View plate.zarr in napari

Two ways to look at a plate.zarr store you've already generated (via `blimp convert
tiff`/`blimp convert nd2`'s NGFF output): a fast, full-resolution overview of the whole
plate laid out at its true row/column positions, and a detailed per-well view with
segmentation labels, point-object tables, and FOV boundaries.

Set `PLATE_PATH` in the cell below to point at your own plate.zarr -- this notebook
doesn't ship with one.

Needs `blimp` importable alongside napari and ngio -- from the `napari-feature-classifier`
environment, run from wherever your own blimp checkout lives:

```
pip install -e /path/to/blimp --no-deps
```

In [ ]:
from pathlib import Path

import napari
from ngio import open_ome_zarr_plate, open_ome_zarr_container

from blimp.napari_utils import add_blimp_napari_methods

PLATE_PATH = Path("/Users/z3532965/Images/example_ngff_creation/plate.zarr")

plate = open_ome_zarr_plate(store=str(PLATE_PATH), mode="r")
wells = plate.wells_paths()
print("Wells with data:", wells)

WELL = wells[0]  # change to e.g. "C/09" to pick a different well
print("Viewing well:", WELL)

## Whole plate at once, laid out like a physical plate

`viewer.add_plate(PLATE_PATH)` builds a lazy, full-resolution pyramid for every
populated well at its true row/column grid position and adds it to the viewer --
one `Image` layer per channel, a `Well_ROI_table` outline for every well (visible by
default), a `Labels` layer per label found on any well (hidden by default), and an
`FOV_ROI_table` outline for every field of view (hidden by default -- turn it on once
zoomed into a single well). Point-object tables are not included here -- that level of
per-object detail belongs in the per-well view below.

This stays fast and full resolution regardless of how sparse the plate is:
`blimp.ome_ngff.plate.build_plate_pyramid` builds the canvas as a `dask.array.zeros`
placeholder (defined analytically -- dask never touches individual chunks just to
construct one) and overlays only populated wells' real data via chunk-aligned
assignment, so cost scales with how many wells actually have data, not with the
plate's declared grid size. A small gap (5% of tile size, computed per pyramid level)
is left between adjacent wells so touching wells stay visually distinct.

In [ ]:
plate_viewer = add_blimp_napari_methods(napari.Viewer())
plate_viewer.add_plate(str(PLATE_PATH))

## Per-well detail: image, labels, measurements, ROIs

Hands the well's image group to the `napari-ome-zarr` plugin reader directly.
OME-NGFF's multiscale pyramid and channel colors are core spec, not a blimp-specific
convention -- the plugin already reads both correctly and lazily (dask-backed, so
napari picks whichever pyramid level fits the current zoom instead of decoding the
full-resolution array up front). It even auto-discovers any labels as plain `Labels`
layers, since OME-NGFF labels are core spec too -- see the next cell for attaching
their measurements.

In [ ]:
for kind in ("mip", "stack"):
    image_group_path = PLATE_PATH / WELL / kind
    if image_group_path.exists():
        break
else:
    raise FileNotFoundError(f"No mip or stack image found for well {WELL}")

viewer = add_blimp_napari_methods(napari.Viewer())
layers = viewer.open(str(image_group_path), plugin="napari-ome-zarr")
print(f"Opened '{kind}':", [layer.name for layer in layers])

## Add measurements, point objects, and FOV boundaries

The cell above's `Labels` layers (if any) are bare pixel data -- napari-ome-zarr has no
idea blimp attached a linked `FeatureTable` to each one, since that's a blimp/Fractal/ngio
convention, not core OME-NGFF spec. Rather than reading the label a second time via
`blimp.napari_utils.add_labels_with_measurements`, attach each one's measurements
directly onto the layer already loaded above.

Point-object tables (spots/blobs with no stable pixel identity -- see
`blimp.ome_ngff.labels._write_well_points`) and FOV boundaries genuinely have no core-spec
equivalent at all, so those still go through blimp's own helpers. A well with none of
these just adds nothing here.

In [ ]:
container = open_ome_zarr_container(str(image_group_path))

for layer in layers:
    if isinstance(layer, napari.layers.Labels):
        label_name = layer.name.rsplit("/", 1)[-1]  # napari-ome-zarr nests it under "labels/labels/<name>"
        table_name = f"{label_name}_features"
        if table_name in container.list_tables():
            layer.features = container.get_feature_table(table_name).dataframe.reset_index()
            print(f"Attached {len(layer.features)} rows of measurements to '{layer.name}'")

for table_name in container.list_tables():
    if container.get_table(table_name).table_type() == "generic_roi_table":
        viewer.add_points_with_measurements(image_group_path, table_name)

if "FOV_ROI_table" in container.list_tables():
    viewer.add_rois(image_group_path)